In [ ]:
import os
import sys
import re
import glob
import json
import numpy as np
import torch
import matplotlib.pyplot as plt
from google.colab import drive

In [ ]:
%cd /content  # place toi sur la machine virtuelle colab 
!rm -rf mtt-distillation # supprimer le dossier mtt-distillation sil existe
!git clone https://github.com/GeorgeCazenavette/mtt-distillation.git
!pip install -q wandb kornia  # on a utiliser cette commande pour installer la bibiotheque koria et wandb demander dans requirements.text

/content
Cloning into 'mtt-distillation'...
remote: Enumerating objects: 77, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 77 (delta 36), reused 25 (delta 25), pack-reused 33 (from 1)
Receiving objects: 100% (77/77), 39.50 MiB | 49.50 MiB/s, done.
Resolving deltas: 100% (37/37), done.


In [ ]:
drive.mount('/content/drive')
os.environ["WANDB_MODE"] = "offline" #pour ne pas utiliser wandb en ligne et ne pas envoyer les donnes sur le cloud 
os.environ["WANDB_SILENT"] = "true"# pour ne pas affichier les messages de wandb dans la console
os.environ["WANDB_API_KEY"] = "dummy"# pour ne pas envoyer les donnes sur le cloud et ne pas utiliser une cle api valide 
print("WANDB_MODE =", os.environ["WANDB_MODE"], "(doit afficher 'offline')")

WANDB_MODE = offline (doit afficher 'offline')


In [ ]:
%cd /content/mtt-distillation
drive.mount('/content/drive', force_remount=True)
assert os.path.exists("/content/drive/MyDrive"), " Drive non monté."
print(" Drive monté.")

os.makedirs("/content/drive/MyDrive/mtt_buffers/CIFAR10/ConvNet", exist_ok=True)

# Test d'écriture
test_file = "/content/drive/MyDrive/mtt_buffers/.write_test"
try:
    with open(test_file, "w") as f: f.write("ok")
    os.remove(test_file)
    print(" Drive accessible en écriture.")
except Exception as e:
    raise RuntimeError(f" Drive non accessible en écriture : {e}")

!python buffer.py \
  --dataset=CIFAR10 \
  --model=ConvNet \
  --train_epochs=50 \
  --num_experts=100 \
  --zca \
  --save_interval=1 \
  --buffer_path=/content/drive/MyDrive/mtt_buffers \
  --data_path=/content/drive/MyDrive/mtt_data \
  2>&1 | tee /content/drive/MyDrive/mtt_buffers/buffer_log.txt
# Vérification finale
saved = glob.glob("/content/drive/MyDrive/mtt_buffers/CIFAR10/ConvNet/*.pt")
log_ok = os.path.exists("/content/drive/MyDrive/mtt_buffers/buffer_log.txt")
if saved and log_ok:
    print(f"\n {len(saved)} buffer(s) + log sauvegardés sur Drive.")
else:
    print(f"\n Problème : {len(saved)} buffers, log présent : {log_ok}")

Mounted at /content/drive
✅ Drive monté.
✅ Drive accessible en écriture.
100%|██████████| 170M/170M [00:05<00:00, 29.0MB/s] 
Train ZCA
100%|██████████| 50000/50000 [00:06<00:00, 7539.53it/s]
Test ZCA
100%|██████████| 10000/10000 [00:01<00:00, 7698.41it/s]
Hyper-parameters: 
 {'dataset': 'CIFAR10', 'subset': 'imagenette', 'model': 'ConvNet', 'res': 128, 'num_experts': 100, 'lr_teacher': 0.01, 'batch_train': 256, 'batch_real': 256, 'dsa': True, 'dsa_strategy': 'color_crop_cutout_flip_scale_rotate', 'data_path': '/content/drive/MyDrive/mtt_data', 'buffer_path': '/content/drive/MyDrive/mtt_buffers', 'train_epochs': 50, 'zca': True, 'decay': False, 'mom': 0, 'l2': 0, 'save_interval': 1, 'device': 'cuda', 'dsa_param': <utils.ParamDiffAug object at 0x7920e47de960>, 'zca_trans': ZCAWhitening()}
BUILDING DATASET
  0%|          | 0/50000 [00:00<?, ?it/s]/content/mtt-distillation/buffer.py:42: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or 

##Distillation CIFAR-10 ipc=1 

In [ ]:
%cd /content/mtt-distillation
# WANDB offline
os.environ["WANDB_MODE"]   = "offline"
os.environ["WANDB_SILENT"] = "true"
os.environ["WANDB_API_KEY"] = "dummy"
print("WANDB_MODE =", os.environ["WANDB_MODE"])

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount('/content/drive', force_remount=True)
assert os.path.exists("/content/drive/MyDrive"), " Drive non monté."
print(" Drive monté.")

# Patch sur distill.py (pour permettre WANDB offline)
with open("distill.py", "r") as f:
    content = f.read()
old_line = 'save_dir = os.path.join(".", "logged_files", args.dataset, wandb.run.name)'
new_line = ('run_name = wandb.run.name if (wandb.run and wandb.run.name) else "offline_run"\n'
            '                save_dir = os.path.join(".", "logged_files", args.dataset, run_name)')
if old_line in content:
    content = content.replace(old_line, new_line)
    with open("distill.py", "w") as f: f.write(content)
    print(" Patch appliqué.")
elif "offline_run" in content:
    print(" Patch déjà appliqué.")
else:
    raise RuntimeError(" Ligne cible introuvable dans distill.py.")

# Vérifier les buffers
buffer_dir = "/content/drive/MyDrive/mtt_buffers/CIFAR10/ConvNet"
buffers = glob.glob(os.path.join(buffer_dir, "*.pt"))
assert len(buffers) > 0, f" Aucun buffer dans {buffer_dir}."
print(f" {len(buffers)} buffer(s) expert(s) trouvés.")

#  SORTIE DÉDIÉE pour ipc=1
output_dir = "/content/drive/MyDrive/mtt_distilled_output_ipc1"
os.makedirs(output_dir, exist_ok=True)
!rm -rf /content/mtt-distillation/logged_files
!ln -sf {output_dir} /content/mtt-distillation/logged_files
target = os.path.realpath("/content/mtt-distillation/logged_files")
assert "/content/drive" in target, f" Symlink mal configuré : {target}"
print(f" Symlink OK → {target}")


!python distill.py \
  --dataset=CIFAR10 \
  --ipc=1 \
  --Iteration=10000 \
  --eval_it=500 \
  --num_eval=3 \
  --epoch_eval_train=1000 \
  --syn_steps=50 \
  --expert_epochs=2 \
  --max_start_epoch=2 \
  --zca \
  --lr_img=1000 \
  --lr_lr=1e-07 \
  --lr_teacher=0.01 \
  --buffer_path=/content/drive/MyDrive/mtt_buffers \
  --data_path=/content/drive/MyDrive/mtt_data \
  2>&1 | tee {output_dir}/distill_log.txt

# Vérification finale
ckpts = glob.glob(f"{output_dir}/CIFAR10/**/images_*.pt", recursive=True)
log_ok = os.path.exists(f"{output_dir}/distill_log.txt")
if ckpts and log_ok:
    print(f"\n ipc=1 terminé : {len(ckpts)} checkpoint(s) + log sauvegardés.")
else:
    print(f"\nProblème : {len(ckpts)} checkpoints, log : {log_ok}")

/content/mtt-distillation
WANDB_MODE = offline
✅ Drive monté.
✅ Patch appliqué.
✅ 100 buffer(s) expert(s) trouvés.
✅ Symlink OK → /content/drive/MyDrive/mtt_distilled_output_ipc1
CUDNN STATUS: True
Train ZCA
100%|██████████| 50000/50000 [00:06<00:00, 7193.89it/s]
Test ZCA
100%|██████████| 10000/10000 [00:01<00:00, 7550.72it/s]
Hyper-parameters: 
 {'_wandb': {'cli_version': '0.26.1', 'python_version': '3.12.13', 't': {'4': '3.12.13', '5': '0.26.1', '12': '0.26.1', '13': 'linux-x86_64', '1': [1, 41, 71, 79], '3': [4, 16]}, 'm': []}, 'dataset': 'CIFAR10', 'subset': 'imagenette', 'model': 'ConvNet', 'res': 128, 'ipc': 1, 'eval_mode': 'S', 'num_eval': 3, 'eval_it': 500, 'epoch_eval_train': 1000, 'Iteration': 10000, 'lr_img': 1000, 'lr_lr': 1e-07, 'lr_teacher': 0.01, 'lr_init': 0.01, 'batch_real': 256, 'batch_syn': 10, 'batch_train': 256, 'pix_init': 'real', 'dsa': True, 'dsa_strategy': 'color_crop_cutout_flip_scale_rotate', 'data_path': '/content/drive/MyDrive/mtt_data', 'buffer_path': '/co

Distillation CIFAR-10 ipc=10

In [ ]:
%cd /content/mtt-distillation
os.environ["WANDB_MODE"]   = "offline"
os.environ["WANDB_SILENT"] = "true"
os.environ["WANDB_API_KEY"] = "dummy"
print("WANDB_MODE =", os.environ["WANDB_MODE"])
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount('/content/drive', force_remount=True)
assert os.path.exists("/content/drive/MyDrive"), " Drive non monté."
print(" Drive monté.")

# Patch distill.py (idempotent — ne refait rien si déjà patché)
with open("distill.py", "r") as f:
    content = f.read()
old_line = 'save_dir = os.path.join(".", "logged_files", args.dataset, wandb.run.name)'
new_line = ('run_name = wandb.run.name if (wandb.run and wandb.run.name) else "offline_run"\n'
            '                save_dir = os.path.join(".", "logged_files", args.dataset, run_name)')
if old_line in content:
    content = content.replace(old_line, new_line)
    with open("distill.py", "w") as f: f.write(content)
    print(" Patch appliqué.")
elif "offline_run" in content:
    print(" Patch déjà appliqué.")
else:
    raise RuntimeError(" Ligne cible introuvable.")

buffer_dir = "/content/drive/MyDrive/mtt_buffers/CIFAR10/ConvNet"
buffers = glob.glob(os.path.join(buffer_dir, "*.pt"))
assert len(buffers) > 0, f" Aucun buffer dans {buffer_dir}."
print(f" {len(buffers)} buffer(s) expert(s) trouvés.")

#  SORTIE DÉDIÉE pour ipc=10
output_dir = "/content/drive/MyDrive/mtt_distilled_output_ipc10"
os.makedirs(output_dir, exist_ok=True)
!rm -rf /content/mtt-distillation/logged_files
!ln -sf {output_dir} /content/mtt-distillation/logged_files
target = os.path.realpath("/content/mtt-distillation/logged_files")
assert "/content/drive" in target, f" Symlink mal configuré : {target}"
print(f" Symlink OK → {target}")

# Hyperparams papier CIFAR-10 @ 10 ipc avec ZCA → cible 65.3%
!python distill.py \
  --dataset=CIFAR10 \
  --ipc=10 \
  --Iteration=10000 \
  --eval_it=500 \
  --num_eval=3 \
  --epoch_eval_train=1000 \
  --syn_steps=50 \
  --expert_epochs=2 \
  --max_start_epoch=20 \
  --zca \
  --lr_img=1000 \
  --lr_lr=1e-05 \
  --lr_teacher=0.01 \
  --buffer_path=/content/drive/MyDrive/mtt_buffers \
  --data_path=/content/drive/MyDrive/mtt_data \
  2>&1 | tee {output_dir}/distill_log.txt

ckpts = glob.glob(f"{output_dir}/CIFAR10/**/images_*.pt", recursive=True)
log_ok = os.path.exists(f"{output_dir}/distill_log.txt")
if ckpts and log_ok:
    print(f"\n ipc=10 terminé : {len(ckpts)} checkpoint(s) + log sauvegardés.")
else:
    print(f"\n Problème : {len(ckpts)} checkpoints, log : {log_ok}")

/content/mtt-distillation
WANDB_MODE = offline
✅ Drive monté.
✅ Patch déjà appliqué.
✅ 100 buffer(s) expert(s) trouvés.
✅ Symlink OK → /content/drive/MyDrive/mtt_distilled_output_ipc10
CUDNN STATUS: True
Train ZCA
100%|██████████| 50000/50000 [00:06<00:00, 7500.95it/s]
Test ZCA
100%|██████████| 10000/10000 [00:01<00:00, 8099.55it/s]
Hyper-parameters: 
 {'_wandb': {'m': [], 'cli_version': '0.26.1', 'python_version': '3.12.13', 't': {'13': 'linux-x86_64', '1': [1, 41, 71, 79], '3': [4, 16], '4': '3.12.13', '5': '0.26.1', '12': '0.26.1'}}, 'dataset': 'CIFAR10', 'subset': 'imagenette', 'model': 'ConvNet', 'res': 128, 'ipc': 10, 'eval_mode': 'S', 'num_eval': 3, 'eval_it': 500, 'epoch_eval_train': 1000, 'Iteration': 10000, 'lr_img': 1000, 'lr_lr': 1e-05, 'lr_teacher': 0.01, 'lr_init': 0.01, 'batch_real': 256, 'batch_syn': 100, 'batch_train': 256, 'pix_init': 'real', 'dsa': True, 'dsa_strategy': 'color_crop_cutout_flip_scale_rotate', 'data_path': '/content/drive/MyDrive/mtt_data', 'buffer_pat

Distillation CIFAR-10 ipc=50

In [ ]:
%cd /content/mtt-distillation
os.environ["WANDB_MODE"]   = "offline"
os.environ["WANDB_SILENT"] = "true"
os.environ["WANDB_API_KEY"] = "dummy"
print("WANDB_MODE =", os.environ["WANDB_MODE"])

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount('/content/drive', force_remount=True)
assert os.path.exists("/content/drive/MyDrive"), " Drive non monté."
print(" Drive monté.")

# Patch distill.py (idempotent)
with open("distill.py", "r") as f:
    content = f.read()
old_line = 'save_dir = os.path.join(".", "logged_files", args.dataset, wandb.run.name)'
new_line = ('run_name = wandb.run.name if (wandb.run and wandb.run.name) else "offline_run"\n'
            '                save_dir = os.path.join(".", "logged_files", args.dataset, run_name)')
if old_line in content:
    content = content.replace(old_line, new_line)
    with open("distill.py", "w") as f: f.write(content)
    print(" Patch appliqué.")
elif "offline_run" in content:
    print(" Patch déjà appliqué.")
else:
    raise RuntimeError(" Ligne cible introuvable.")

buffer_dir = "/content/drive/MyDrive/mtt_buffers/CIFAR10/ConvNet"
buffers = glob.glob(os.path.join(buffer_dir, "*.pt"))
assert len(buffers) > 0, f" Aucun buffer dans {buffer_dir}."
print(f" {len(buffers)} buffer(s) expert(s) trouvés.")

#  SORTIE DÉDIÉE pour ipc=50
output_dir = "/content/drive/MyDrive/mtt_distilled_output_ipc50"
os.makedirs(output_dir, exist_ok=True)
!rm -rf /content/mtt-distillation/logged_files
!ln -sf {output_dir} /content/mtt-distillation/logged_files
target = os.path.realpath("/content/mtt-distillation/logged_files")
assert "/content/drive" in target, f" Symlink mal configuré : {target}"
print(f" Symlink OK → {target}")


!python distill.py \
  --dataset=CIFAR10 \
  --ipc=50 \
  --Iteration=10000 \
  --eval_it=500 \
  --num_eval=3 \
  --epoch_eval_train=1000 \
  --syn_steps=30 \
  --expert_epochs=2 \
  --max_start_epoch=40 \
  --zca \
  --lr_img=1000 \
  --lr_lr=1e-05 \
  --lr_teacher=0.01 \
  --batch_syn=125 \
  --buffer_path=/content/drive/MyDrive/mtt_buffers \
  --data_path=/content/drive/MyDrive/mtt_data \
  2>&1 | tee {output_dir}/distill_log.txt

ckpts = glob.glob(f"{output_dir}/CIFAR10/**/images_*.pt", recursive=True)
log_ok = os.path.exists(f"{output_dir}/distill_log.txt")
if ckpts and log_ok:
    print(f"\n ipc=50 terminé : {len(ckpts)} checkpoint(s) + log sauvegardés.")
else:
    print(f"\n Problème : {len(ckpts)} checkpoints, log : {log_ok}")

/content/mtt-distillation
WANDB_MODE = offline
✅ Drive monté.
✅ Patch déjà appliqué.
✅ 100 buffer(s) expert(s) trouvés.
✅ Symlink OK → /content/drive/MyDrive/mtt_distilled_output_ipc50
CUDNN STATUS: True
Train ZCA
100%|██████████| 50000/50000 [00:06<00:00, 7230.89it/s]
Test ZCA
100%|██████████| 10000/10000 [00:01<00:00, 7068.39it/s]
Hyper-parameters: 
 {'_wandb': {'cli_version': '0.26.1', 'python_version': '3.12.13', 't': {'1': [1, 41, 71, 79], '3': [4, 16], '4': '3.12.13', '5': '0.26.1', '12': '0.26.1', '13': 'linux-x86_64'}, 'm': []}, 'dataset': 'CIFAR10', 'subset': 'imagenette', 'model': 'ConvNet', 'res': 128, 'ipc': 50, 'eval_mode': 'S', 'num_eval': 3, 'eval_it': 500, 'epoch_eval_train': 1000, 'Iteration': 10000, 'lr_img': 1000, 'lr_lr': 1e-05, 'lr_teacher': 0.01, 'lr_init': 0.01, 'batch_real': 256, 'batch_syn': 125, 'batch_train': 256, 'pix_init': 'real', 'dsa': True, 'dsa_strategy': 'color_crop_cutout_flip_scale_rotate', 'data_path': '/content/drive/MyDrive/mtt_data', 'buffer_pat